# 3: Preprocessing & Normalization
**Goal:** Clean arXiv text, compare word tokenizers, and build a finalized normalization pipeline.

---

## Imports

In [2]:
import re
import time
import pandas as pd
import numpy as np
from collections import Counter

In [3]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, TreebankWordTokenizer, TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

for resource in ["punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

## Loading Sample

In [5]:
df_full = pd.read_parquet("../ext/sample_50k.parquet")

SAMPLE_SIZE = 50_000
RANDOM_SEED = 42
df_sample = df_full.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Working sample: {len(df_sample):,} rows")
print(f"Columns: {list(df_sample.columns)}")

Working sample: 50,000 rows
Columns: ['id', 'title', 'abstract', 'categories']


## Text Cleaning

arXiv abstracts contain LaTeX artifacts from the original TeX source files. These need to be removed before tokenization, otherwise tokenizers split on math symbols and produce noise tokens like `$`, `{`, `\\alpha`.

**What we clean:**
- Inline math: `$x^2 + y^2$`
- LaTeX commands with arguments: `\cite{vaswani2017}`, `\textbf{important}`
- Bare LaTeX commands: `\alpha`, `\beta`, `\rightarrow`
- Leftover curly braces
- Excessive whitespace and newlines

In [29]:
#ordered list of (pattern, replacement)
#order matters
LATEX_PATTERNS = [
    (r"\$[^$]*\$",              ""),   # inline math $...$
    (r"\\[a-zA-Z]+\{[^}]*\}",  ""),   # \command{arg}
    (r"\\[a-zA-Z]+",            ""),   # bare \command
    (r"\{[^}]*\}",              ""),   # leftover {braces}
    (r"\s+",                    " "),   # collapse whitespace
]

def clean_text(text: str) -> str:
    """Remove LaTeX artifacts and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    for pattern, replacement in LATEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    text = text.encode("ascii", errors="ignore").decode()  # remove non-ASCII
    return text.strip()

#demo
raw_example = df_sample["abstract"].iloc[0]
clean_example = clean_text(raw_example)

print("RAW:")
print(raw_example[:400])
print("\nCLEANED:")
print(clean_example[:400])

RAW:
Specific heat measurements of hydrogenated amorphous silicon prepared by
hot-wire chemical vapor deposition show a large density of two-level systems at
low temperature. Annealing at 200 {\deg}C, well below the growth temperature,
does not significantly affect the already-low internal friction or the sound
velocity, but irreversibly reduces the non-Debye specific heat by an order of
magnitude at 2

CLEANED:
Specific heat measurements of hydrogenated amorphous silicon prepared by hot-wire chemical vapor deposition show a large density of two-level systems at low temperature. Annealing at 200 C, well below the growth temperature, does not significantly affect the already-low internal friction or the sound velocity, but irreversibly reduces the non-Debye specific heat by an order of magnitude at 2 K, in


In [28]:
#applying the cleaning to full sample
df_sample["abstract_clean"] = df_sample["abstract"].apply(clean_text)
df_sample["title_clean"] = df_sample["title"].apply(clean_text)

#how many abstracts were actually changed:
changed = (df_sample["abstract"] != df_sample["abstract_clean"]).sum()
print(f"Abstracts affected by cleaning: {changed:,} / {len(df_sample):,} ({changed/len(df_sample)*100:.1f}%)")

Abstracts affected by cleaning: 45,889 / 50,000 (91.8%)


## Sentence Tokenization

We use NLTK's `sent_tokenize` (Punkt algorithm) rather than a regex approach because:
- It handles scientific abbreviations like `et al.`, `Fig.`, `Eq.` correctly
- A naive regex on `.` would incorrectly split `e.g.` or `i.e.` into two sentences
- It's been trained on a large corpus and handles edge cases we'd miss manually

In [27]:
#demo on a single abstract
sentences = sent_tokenize(clean_example)
print(f"Sentences found: {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"[{i}] {s}")

Sentences found: 5
[1] Specific heat measurements of hydrogenated amorphous silicon prepared by hot-wire chemical vapor deposition show a large density of two-level systems at low temperature.
[2] Annealing at 200 C, well below the growth temperature, does not significantly affect the already-low internal friction or the sound velocity, but irreversibly reduces the non-Debye specific heat by an order of magnitude at 2 K, indicating a large reduction of the density of two-level systems.
[3] Comparison of the specific heat to the internal friction suggests that the two-level systems are uncharacteristically decoupled from acoustic waves, both before and after annealing.
[4] Analysis yields an anomalously low value of the coupling constant, which increases upon annealing but still remains anomalously low.
[5] The results suggest that the coupling constant value is lowered by the presence of hydrogen.


In [26]:
#applying to full sample
start = time.time()
df_sample["sentences"]     = df_sample["abstract_clean"].apply(sent_tokenize)
df_sample["num_sentences"] = df_sample["sentences"].apply(len)
print(f"Sentence tokenization done in {time.time() - start:.2f}s")

print("\nSentences per abstract:")
print(df_sample["num_sentences"].describe().round(2).to_string())

Sentence tokenization done in 2.48s

Sentences per abstract:
count    50000.00
mean         6.13
std          2.74
min          1.00
25%          4.00
50%          6.00
75%          8.00
max         25.00


## Word Tokenizer Comparison

We benchmark three NLTK tokenizers to pick the best fit for academic text:

| Tokenizer | Designed For | Notes |
|-----------|-------------|-------|
| `word_tokenize` | General English | Uses Punkt + regex, handles punctuation well |
| `TreebankWordTokenizer` | Formal written text | Penn Treebank conventions, fast, good baseline |
| `TweetTokenizer` | Social media | Included as a contrast — not expected to excel here |

In [25]:
#inspecting tokens side by side on one abstract
sample_abstract = df_sample["abstract_clean"].iloc[0]

tokenizers = {
    "word_tokenize"        : lambda t: word_tokenize(t),
    "TreebankWordTokenizer" : lambda t: TreebankWordTokenizer().tokenize(t),
    "TweetTokenizer"        : lambda t: TweetTokenizer().tokenize(t),
}

print(f"Abstract (first 300 chars):\n{sample_abstract[:300]}...\n")

for name, fn in tokenizers.items():
    tokens = fn(sample_abstract)
    print(f"{name}:")
    print(f"  First 20 tokens : {tokens[:20]}")
    print(f"  Total tokens    : {len(tokens)}\n")

Abstract (first 300 chars):
Specific heat measurements of hydrogenated amorphous silicon prepared by hot-wire chemical vapor deposition show a large density of two-level systems at low temperature. Annealing at 200 C, well below the growth temperature, does not significantly affect the already-low internal friction or the soun...

word_tokenize:
  First 20 tokens : ['Specific', 'heat', 'measurements', 'of', 'hydrogenated', 'amorphous', 'silicon', 'prepared', 'by', 'hot-wire', 'chemical', 'vapor', 'deposition', 'show', 'a', 'large', 'density', 'of', 'two-level', 'systems']
  Total tokens    : 139

TreebankWordTokenizer:
  First 20 tokens : ['Specific', 'heat', 'measurements', 'of', 'hydrogenated', 'amorphous', 'silicon', 'prepared', 'by', 'hot-wire', 'chemical', 'vapor', 'deposition', 'show', 'a', 'large', 'density', 'of', 'two-level', 'systems']
  Total tokens    : 135

TweetTokenizer:
  First 20 tokens : ['Specific', 'heat', 'measurements', 'of', 'hydrogenated', 'amorphous', 'silicon'

Benchmarking:

In [15]:
benchmark_results = {}

for name, fn in tokenizers.items():
    start = time.time()
    counts = df_sample["abstract_clean"].apply(lambda x: len(fn(x)))
    elapsed = time.time() - start

    benchmark_results[name] = {
        "time_sec"   : round(elapsed, 2),
        "mean_tokens": round(counts.mean(), 1),
        "median"     : round(counts.median(), 1),
        "std"        : round(counts.std(), 1),
        "min"        : int(counts.min()),
        "max"        : int(counts.max()),
    }

    print(f"{name}:")
    print(f"Time: {elapsed:.2f}s for {len(df_sample):,} abstracts")
    print(f"Mean: {counts.mean():.1f}  |  Median: {counts.median():.1f}  |  Std: {counts.std():.1f}\n")

bench_df = pd.DataFrame(benchmark_results).T
print("\nSummary table:")
bench_df

word_tokenize:
Time: 17.21s for 50,000 abstracts
Mean: 159.4  |  Median: 155.0  |  Std: 70.5

TreebankWordTokenizer:
Time: 9.61s for 50,000 abstracts
Mean: 154.5  |  Median: 151.0  |  Std: 68.2

TweetTokenizer:
Time: 18.45s for 50,000 abstracts
Mean: 161.2  |  Median: 157.0  |  Std: 71.5


Summary table:


,time_sec,mean_tokens,median,std,min,max
word_tokenize,17.21,159.4,155.0,70.5,4.0,632.0
TreebankWordTokenizer,9.61,154.5,151.0,68.2,4.0,611.0
TweetTokenizer,18.45,161.2,157.0,71.5,4.0,636.0


### Tokenizer Decision

We choose **`word_tokenize`** as our tokenizer for the following reasons:

- **Best fit for academic text:** Handles punctuation, hyphens, and contractions in a way that preserves scientific terms (e.g., `state-of-the-art` stays meaningful)
- **Punkt integration:** Uses the same sentence-level model as `sent_tokenize`, so behavior is consistent across our pipeline
- **`TreebankWordTokenizer`** is slightly faster but splits hyphenated terms more aggressively, which loses domain-specific compound terms common in STEM papers
- **`TweetTokenizer`** is designed for informal short text and produces the most tokens — not appropriate for formal academic writing

## Token Normalization

### Why lemmatization over stemming?

| Method | Example | Output | Issue |
|--------|---------|--------|-------|
| Stemming (Porter) | `running`, `studies` | `run`, `studi` | Produces non-words (`studi` is not a real word) |
| Lemmatization (WordNet) | `running`, `studies` | `run`, `study` | Produces real dictionary words |

For academic text, lemmatization is the correct choice because:
- Scientific terms are often morphologically complex — stemming can mangle them
- Interpretability matters — we want to be able to read our vocabulary and understand it
- WordNet lemmatization is conservative; it only reduces to a valid dictionary form

In [16]:
#side-by-side comparison: stemming vs lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "studies", "algorithms", "embeddings",
              "matrices", "classifiers", "optimization", "regularization"]

print(f"{'Word':<20} {'Stem':<20} {'Lemma':<20}")
print("-" * 60)
for w in test_words:
    print(f"{w:<20} {stemmer.stem(w):<20} {lemmatizer.lemmatize(w):<20}")

Word                 Stem                 Lemma               
------------------------------------------------------------
running              run                  running             
studies              studi                study               
algorithms           algorithm            algorithm           
embeddings           embed                embeddings          
matrices             matric               matrix              
classifiers          classifi             classifier          
optimization         optim                optimization        
regularization       regular              regularization      


In [24]:
#normalization pipeline
STOP_WORDS = set(stopwords.words("english"))

def normalize_tokens(text: str) -> list:
    """
    Full normalization pipeline:
      1. word_tokenize
      2. Lowercase
      3. Keep only alphabetic tokens (removes punctuation and numbers)
      4. Remove stopwords
      5. Lemmatize with WordNet

    We use lemmatization over stemming because stemming produces
    non-dictionary forms that are hard to interpret and can mangle
    scientific terminology (e.g. 'matrices' -> 'matric' under Porter).
    """
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    tokens = [t for t in tokens if t.isalpha()] #remove punctuation & numbers
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

#demo on a single abstract
sample_clean = df_sample["abstract_clean"].iloc[0]
raw_tokens   = word_tokenize(sample_clean)
norm_tokens  = normalize_tokens(sample_clean)

print(f"Raw tokens    ({len(raw_tokens)})  : {raw_tokens[:15]}")
print(f"Norm tokens   ({len(norm_tokens)}) : {norm_tokens[:15]}")
print(f"Reduction     : {(1 - len(norm_tokens)/len(raw_tokens))*100:.1f}%")

Raw tokens    (139)  : ['Specific', 'heat', 'measurements', 'of', 'hydrogenated', 'amorphous', 'silicon', 'prepared', 'by', 'hot-wire', 'chemical', 'vapor', 'deposition', 'show', 'a']
Norm tokens   (73) : ['specific', 'heat', 'measurement', 'hydrogenated', 'amorphous', 'silicon', 'prepared', 'chemical', 'vapor', 'deposition', 'show', 'large', 'density', 'system', 'low']
Reduction     : 47.5%


In [19]:
#applying to full sample
start = time.time()
df_sample["normalized_tokens"] = df_sample["abstract_clean"].apply(normalize_tokens)
df_sample["normalized_len"]    = df_sample["normalized_tokens"].apply(len)
print(f"Normalization done in {time.time() - start:.2f}s")

#before vs after
raw_counts = df_sample["abstract_clean"].apply(lambda x: len(word_tokenize(x)))
print(f"\nMean raw tokens     : {raw_counts.mean():.1f}")
print(f"Mean norm tokens    : {df_sample['normalized_len'].mean():.1f}")
print(f"Token reduction     : {(1 - df_sample['normalized_len'].mean() / raw_counts.mean()) * 100:.2f}%")

Normalization done in 26.53s

Mean raw tokens     : 159.4
Mean norm tokens    : 81.4
Token reduction     : 48.94%


## Vocab Analysis

In [20]:
#vocabulary size comparison: raw vs normalized
all_raw_tokens  = df_sample["abstract_clean"].apply(word_tokenize).explode()
all_norm_tokens = df_sample["normalized_tokens"].explode()

print(f"Raw vocabulary size        : {all_raw_tokens.nunique():,}")
print(f"Normalized vocabulary size : {all_norm_tokens.nunique():,}")
print(f"Vocabulary reduction       : {(1 - all_norm_tokens.nunique() / all_raw_tokens.nunique()) * 100:.2f}%")

Raw vocabulary size        : 176,695
Normalized vocabulary size : 65,488
Vocabulary reduction       : 62.94%


In [21]:
#top 20 most frequent normalized tokens
vocab = Counter(all_norm_tokens.dropna())
top_tokens = vocab.most_common(20)

print("Top 20 most frequent normalized tokens:")
for token, count in top_tokens:
    print(f"  {token:<25}: {count:,}")

Top 20 most frequent normalized tokens:
  model                    : 39,659
  result                   : 21,759
  method                   : 21,161
  system                   : 20,534
  data                     : 18,809
  show                     : 18,027
  using                    : 16,155
  also                     : 14,446
  study                    : 14,430
  field                    : 13,940
  two                      : 13,603
  paper                    : 12,909
  problem                  : 12,876
  approach                 : 12,848
  state                    : 12,597
  function                 : 12,090
  time                     : 11,856
  quantum                  : 11,529
  network                  : 10,689
  one                      : 10,628


## Saving Preprocessed Sample

In [22]:
#drops list columns before saving (parquet handles them but can cause issues downstream)
#keeping the cleaned text columns and numeric stats; lists get regenerated when needed
save_cols = ["id", "title", "abstract", "categories",
             "title_clean", "abstract_clean",
             "num_sentences", "normalized_len"]

df_save = df_sample[save_cols].copy()
df_save.to_parquet("../ext/sample_50k_preprocessed.parquet", index=False)
print(f"Saved preprocessed sample: {len(df_save):,} rows")
print(f"Columns saved: {list(df_save.columns)}")

Saved preprocessed sample: 50,000 rows
Columns saved: ['id', 'title', 'abstract', 'categories', 'title_clean', 'abstract_clean', 'num_sentences', 'normalized_len']


## Summary

| Step | Choice | Rationale |
|------|--------|-----------|
| **Text cleaning** | Regex-based LaTeX removal | arXiv abstracts have heavy LaTeX artifacts that corrupt tokenization |
| **Sentence tokenizer** | `sent_tokenize` (Punkt) | Handles scientific abbreviations; regex would split `et al.` incorrectly |
| **Word tokenizer** | `word_tokenize` | Best handles academic punctuation and hyphenated terms |
| **Normalization** | Lowercase → alpha-only → stopwords → lemmatize | Reduces vocabulary while keeping meaningful stems |
| **Stemming vs lemmatization** | Lemmatization (WordNet) | Produces real dictionary words; stemming mangles scientific terms |